All code was run on Google Colab (T4 GPU) using Jupyter notebooks. Total runtime is roughly 90 minutes. Epoch count was not tuned. No parameter sensitivity analysis was carried out.

In [ ]:
!pip install tensorboard -q

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device}")

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False, num_workers=2)

cuda


100%|██████████| 170M/170M [00:04<00:00, 39.4MB/s]


In [ ]:
model_alex = models.alexnet(weights=None).to(device)
model_alex.classifier[6] = nn.Linear(4096, 10)
model_alex = model_alex.to(device)

optimizer = optim.Adam(model_alex.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/AlexNet_scratch")

In [ ]:
for epoch in range(10):
    model_alex.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_alex(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f}")
writer.close()

model_alex.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_alex(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"AlexNet Scratch Accuracy: {acc:.2f}%")
torch.save(model_alex.state_dict(), "alexnet_scratch.pth")
results = {"AlexNet_scratch": acc}

Epoch 1/10 - Loss: 1.5259
Epoch 2/10 - Loss: 1.0500
Epoch 3/10 - Loss: 0.8350
Epoch 4/10 - Loss: 0.6936
Epoch 5/10 - Loss: 0.5924
Epoch 6/10 - Loss: 0.5021
Epoch 7/10 - Loss: 0.4330
Epoch 8/10 - Loss: 0.3662
Epoch 9/10 - Loss: 0.3091
Epoch 10/10 - Loss: 0.2626
AlexNet Scratch Accuracy: 82.40%


In [ ]:
model_alex2 = models.alexnet(weights=models.AlexNet_Weights.DEFAULT).to(device)
model_alex2.classifier[6] = nn.Linear(4096, 10)
model_alex2 = model_alex2.to(device)

optimizer = optim.Adam(model_alex2.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/AlexNet_pretrained")

for epoch in range(10):
    model_alex2.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_alex2(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f}")
writer.close()

model_alex2.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_alex2(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"AlexNet Pretrained Accuracy: {acc:.2f}%")
torch.save(model_alex2.state_dict(), "alexnet_pretrained.pth")
results["AlexNet_pretrained"] = acc

Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:02<00:00, 90.3MB/s]


Epoch 1/10 - Loss: 0.5693
Epoch 2/10 - Loss: 0.3124
Epoch 3/10 - Loss: 0.2091
Epoch 4/10 - Loss: 0.1517
Epoch 5/10 - Loss: 0.1151
Epoch 6/10 - Loss: 0.0876
Epoch 7/10 - Loss: 0.0738
Epoch 8/10 - Loss: 0.0671
Epoch 9/10 - Loss: 0.0591
Epoch 10/10 - Loss: 0.0481
AlexNet Pretrained Accuracy: 90.94%


In [ ]:
import pandas as pd
pd.DataFrame(results.items(), columns=["Run", "Test Accuracy"]).to_csv("results.csv", index=False)
from google.colab import files
files.download("results.csv")
files.download("alexnet_scratch.pth")
files.download("alexnet_pretrained.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
transform_mnist = transforms.Compose([
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset_mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist)
testset_mnist  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

trainloader_mnist = torch.utils.data.DataLoader(trainset_mnist, batch_size=64, shuffle=True, num_workers=2)
testloader_mnist  = torch.utils.data.DataLoader(testset_mnist,  batch_size=64, shuffle=False, num_workers=2)
##Lab 0.2.2, using tanh but adopting it for grey scale
model_mnist = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.Tanh(), nn.MaxPool2d(2, 2),
    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 256), nn.Tanh(),
    nn.Linear(256, 10)
).to(device)

optimizer = optim.Adam(model_mnist.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/MNIST_CNN")

for epoch in range(10):
    model_mnist.train()
    running_loss = 0.0
    for inputs, labels in trainloader_mnist:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_mnist(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader_mnist)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f}")
writer.close()

model_mnist.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader_mnist:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_mnist(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"MNIST Accuracy: {acc:.2f}%")
torch.save(model_mnist.state_dict(), "mnist_cnn.pth")
results["MNIST_CNN"] = acc

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 491kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.9MB/s]


Epoch 1/10 - Loss: 0.3552
Epoch 2/10 - Loss: 0.1065
Epoch 3/10 - Loss: 0.0685
Epoch 4/10 - Loss: 0.0518
Epoch 5/10 - Loss: 0.0413
Epoch 6/10 - Loss: 0.0345
Epoch 7/10 - Loss: 0.0284
Epoch 8/10 - Loss: 0.0239
Epoch 9/10 - Loss: 0.0210
Epoch 10/10 - Loss: 0.0175
MNIST Accuracy: 98.88%


In [ ]:
transform_svhn = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize(32),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset_svhn = torchvision.datasets.SVHN(root='./data', split='train', download=True, transform=transform_svhn)
testset_svhn  = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform_svhn)

trainloader_svhn = torch.utils.data.DataLoader(trainset_svhn, batch_size=64, shuffle=True, num_workers=2)
testloader_svhn  = torch.utils.data.DataLoader(testset_svhn,  batch_size=64, shuffle=False, num_workers=2)

100%|██████████| 182M/182M [00:01<00:00, 104MB/s]
100%|██████████| 64.3M/64.3M [00:01<00:00, 59.1MB/s]


In [ ]:
optimizer = optim.Adam(model_mnist.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
writer = SummaryWriter("runs/SVHN_transfer")

# using MNIST pretrained weights as starting point for SVHN (transfer learning)
for epoch in range(10):
    model_mnist.train()
    running_loss = 0.0
    for inputs, labels in trainloader_svhn:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model_mnist(inputs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(trainloader_svhn)
    writer.add_scalar("Loss/train", avg_loss, epoch)
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f}")
writer.close()

model_mnist.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader_svhn:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_mnist(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc = 100 * correct / total
print(f"SVHN Transfer Accuracy: {acc:.2f}%")
torch.save(model_mnist.state_dict(), "svhn_transfer.pth")
results["SVHN_transfer"] = acc

Epoch 1/10 - Loss: 1.0916
Epoch 2/10 - Loss: 0.5860
Epoch 3/10 - Loss: 0.4774
Epoch 4/10 - Loss: 0.4182
Epoch 5/10 - Loss: 0.3772
Epoch 6/10 - Loss: 0.3450
Epoch 7/10 - Loss: 0.3181
Epoch 8/10 - Loss: 0.2950
Epoch 9/10 - Loss: 0.2753
Epoch 10/10 - Loss: 0.2552
SVHN Transfer Accuracy: 88.11%


In [ ]:
## Just forgot step 2 before step 3, so re-loading pre-fine tuning weights.
model_mnist.load_state_dict(torch.load("mnist_cnn.pth"))

model_mnist.eval()
correct = total = 0
with torch.no_grad():
    for inputs, labels in testloader_svhn:
        inputs, labels = inputs.to(device), labels.to(device)
        _, predicted = torch.max(model_mnist(inputs), 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
acc_direct = 100 * correct / total
print(f"MNIST model directly on SVHN (no retraining): {acc_direct:.2f}%")
results["MNIST_direct_SVHN"] = acc_direct

MNIST model directly on SVHN (no retraining): 15.39%


In [ ]:
import pandas as pd
pd.DataFrame(results.items(), columns=["Run", "Test Accuracy"]).to_csv("results_task02.csv", index=False)

import shutil
shutil.make_archive("task02_runs", "zip", "runs")

from google.colab import files
files.download("results_task02.csv")
files.download("mnist_cnn.pth")
files.download("svhn_transfer.pth")
files.download("task02_runs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>